# Diabetes 130-US Hospitals

## 1. EDA & problem framing

### Downloading necessary libraries

We start by downloading the necessary libraries to better visualize the data

In [ ]:
import missingno as msno
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

### Dataset loading

Load the data from the sklearn library (Diabetes data)

In [ ]:
import pandas as pd

from sklearn.datasets import fetch_openml
# Option 3 -- Diabetes 130 Hospitals
diabetes = fetch_openml(data_id=4541, as_frame=True)
X, y = diabetes.data, diabetes.target

# 1.1 Problem Definition

#### Goal: Predict whether a patient will be readmitted to the hospital
#### The data comes from 130 US hospitals between 1999-2008; each row is one hospital stay of a patient with diabetes. The same patient could have several stays.


# 1.2 Target 

#### Readmitted has 3 labels, <30; >30 and NO.

In [ ]:
y.value_counts()

#### Check to see there no missing values in the target

In [ ]:
y.isna().sum()

#### We convert it to binary values (1, 0) maning yes and no respectively.

In [ ]:
y = y.map({'NO': 0, '<30': 1, '>30': 1})

#### Check to make sure it works!

In [ ]:
y.value_counts()

# 1.3 Visualize the data

In [ ]:
df = pd.DataFrame(X, columns=diabetes.feature_names)

In [ ]:
df.shape

In [ ]:
df.info()

### We see the data inside each row and as we can see we have some data that has question marks, so we need to change them to NaaNs to have a better view

In [ ]:
df.head(10)

#### We create a list called cols_count that includes the countable features in our dataset. Below we can see the descriptive statistics of said features.

##### Below we can see the stays in the hospitals range between 1 to 14 days.
##### We can see that for each patient, there needs to be AT LEAST 1 lab procedure.

In [ ]:
cols_count = ["time_in_hospital", "num_lab_procedures", "num_procedures", "num_medications", "number_outpatient", "number_emergency", "number_inpatient", "number_diagnoses"]
df[cols_count].describe()

# 1.4 Cleaning the Data

##### We do a missingno matrix and immediately see that there are no missing values, which is strange. After reviewing the data, we can see that there are ?'s in these columns (weight, medical specialty, payer_code and race). 
##### We also saw that there is a 'None' value in the A1Cresult and max_glu_serum which means no test was performed.
##### Moreover, the gender column has 3 stays which are were recorded as Unknown/Invalid.
##### We also check for duplicated in our data and see that there are none.

In [ ]:
msno.matrix(df)

In [ ]:
df.duplicated().sum()

##### Now we replace all the missing values with NaaN.

In [ ]:
df = df.replace('?', np.nan)
df = df.replace('None', np.nan)
df = df.replace('Unknown/Invalid', np.nan)
df.isna().sum()

##### Now that we established that we have a lot of missing values, we will delete the top 5 columns because even if we fill with the mean, median or mode, the analysis will be flawed.

In [ ]:
missing_percentage = df.isna().sum() / len(df) * 100
missing_percentage[missing_percentage > 0].sort_values(ascending=False).round(4)

In [ ]:
df = df.drop(columns=['weight', 'max_glu_serum', 'A1Cresult', 'medical_specialty','payer_code'])

##### Now, we replace diagnostic missing values with "Unknown".

In [ ]:
for col in ["diag_1", "diag_2", "diag_3"]:
    df[col] = df[col].astype("string").fillna("Unknown")

In [ ]:
msno.bar(df)

### We run a df.describe on the diagnoses and see which diagnosis is most repititive. 
#### 1. Based on the information below, the main diagnosis is 428 which refers to heart failure :(
#### 2. Then, the second most common diagnosis is 276 which is disorder of fluid :(
#### 3. Finally, the third most common diagnosis is 250 which is Diabetes Mellitus :(

In [ ]:
df[["diag_1", "diag_2", "diag_3"]].mode()

# 1.5 Variables Distributions 

##### Visualize the binary target distribution in percentages.

In [ ]:
y.value_counts(normalize=True) * 100

In [ ]:
y.value_counts().sort_index().plot.bar()
plt.title("Distribution of the target variable")
plt.xticks([0, 1], ['NO', 'YES'], rotation=0)
plt.ylabel("Number of Stays")
plt.show()

In [ ]:
plt.figure(figsize=(8, 4))
sns.countplot(data=df, x="age", order=sorted(df["age"]), color="coral")
plt.title("Age distribution")
plt.xlabel("Age group")
plt.ylabel("Number of stays")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 4))
sns.histplot(data=df, x="time_in_hospital", discrete=True, color="salmon")
plt.title("Length of hospital stays")
plt.xlabel("Days in hospital")
plt.ylabel("Number of stays")
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(8, 4))
sns.countplot(data=df, x="number_inpatient", color="indigo")
plt.grid(axis='y', linestyle='--', alpha=0.7, color='green')
plt.title("Hospitalizations in the previous year")
plt.xlabel("Number of previous hospitalizations")
plt.ylabel("Number of stays")
plt.tight_layout()
plt.show()

##### Now we look into the patients that have been admitted multiple times and patients that have been admitted only once.
##### To break down the maths, we take the number of patients with several stays and subtract it from the unique patients and then add the total stays for patients with several days, which gives us the total number of admissions (101,766).

In [ ]:
stays_per_patient = df["patient_nbr"].value_counts()

print("Unique patients:", stays_per_patient.size)
print("Patients with several stays:", (stays_per_patient > 1).sum())
print("Total stays for patients with several stays:", stays_per_patient[stays_per_patient > 1].sum())

# 1.6 Splitting the data

##### We split train and test per patient; we import GroupKFold which allows us to split the data whitout separating groups (patient with multiple admissions).
##### We set nsplits = 5 to have an 80/20 split.

In [ ]:
from sklearn.model_selection import GroupKFold

split = GroupKFold(n_splits=5, shuffle=True, random_state=42)

train_set, test_set = next(
    split.split(df, y, groups=df["patient_nbr"])
)

X_train = df.iloc[train_set].copy()
X_test = df.iloc[test_set].copy()

y_train = y.iloc[train_set].copy()
y_test = y.iloc[test_set].copy()

##### We verify the sepratation and since the resulet is set(), we know that both sets don't have a common patient.

In [ ]:
set(X_train["patient_nbr"]) & set(X_test["patient_nbr"])

##### Next, we store the patient_nbr in groups_train for cross validation then delete them because they aren't medical characteristics.

In [ ]:
groups_train = X_train["patient_nbr"]

X_train = X_train.drop(columns=["encounter_id", "patient_nbr"])
X_test = X_test.drop(columns=["encounter_id", "patient_nbr"])

### EDA conclusion

Our goal is to predict whether a hospital stay will be followed by a recorded readmission, regardless of the delay. We plan to make predictions at discharge.

The target is binary: 1 for a recorded readmission (⁠ <30 ⁠ or ⁠ >30 ⁠) and 0 for no recorded readmission (⁠ NO ⁠). About 46% of stays belong to class 1, so the classes are relatively balanced.

The dataset contains 101,766 stays from 71,518 patients. Among these patients, 16,773 have multiple stays. We will therefore split the data by patient to prevent overlap between training and test sets.

Unknown diagnosis codes are represented by an "Unknown" category. Diagnosis codes and other category codes will be treated as categorical variables.

We split the data before fitting any imputation or encoding. Patient and encounter identifiers are excluded from the model's predictors.

# 2. Feature Engineering

##### We do a for loop over each line so we could identify the total number of visits per stay throughout the year before hospitalisation. Then, we devide the number of lab procedures during the encounter by the time spent in hospital to figure out the number of lab procedures per day.

In [ ]:
for data in [X_train, X_test]:
    data["previous_visits"] = (
        data["number_outpatient"]
        + data["number_emergency"]
        + data["number_inpatient"]
    )

    data["lab_procedures_per_day"] = (
        data["num_lab_procedures"] / data["time_in_hospital"]
    )

In [ ]:
X_train[["previous_visits", "lab_procedures_per_day"]].sample(10)

##### Separating the numerical and categorical columns.

In [ ]:
numeric_columns = [
    "time_in_hospital",
    "num_lab_procedures",
    "num_procedures",
    "num_medications",
    "number_outpatient",
    "number_emergency",
    "number_inpatient",
    "number_diagnoses",
    "previous_visits",
    "lab_procedures_per_day"
]

categorical_columns = X_train.columns.drop(numeric_columns).tolist()

In [ ]:
print("Numerical:", numeric_columns)
print("Categorical:", categorical_columns)

##### We set up the pipelines, where the numerical pipeline we set up the imputer and standard scaler, then for the categorical pipeline we set up the imputer and encoder of choice (OneHotEncoder). We chose OneHotEncoder because we wanted to transform the categories to numbers without creating an order between them and avoid data leakage.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

##### We add a ColumnTransformer which is a preprocessor which applies the preprocess into the pipeline.

In [ ]:
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_columns),
    ("categorical", categorical_pipeline, categorical_columns)
])

# 3. Class Imbalance Handling

##### The training classes are more or less balanced. We should keep the original distribution without oversampling or undersampling.
##### We will compare the models by computing the accuracy, precision, recall, F1-score and ROC-AUC.
##### We will also compare class weighting with no weighting during the model selection and rather during the model training, while also using cross-validation on the development set.

In [ ]:
y_train.value_counts(normalize=True) * 100

## 4. Modelling progression

##### Now it's time to set up the 4 type of models we want to use, we decided to use Logistic Regression, Random Forest, CatBoost and LightGBM.
The choice behind choosing these 4 types of models is because they use different approaches to learn from the data. 

Logistic Regression has a simple baseline to predict readmission and understand correlation between features.

Random Forest can capture the nonlinear relationships.

LightGBM builds trees to improve which allows us to test whether boosting improves the predictions.

CatBoost handles categorical features directly, making it suitable for our dataset with many categorical variables, including diagnosis codes.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier


logistic = LogisticRegression(max_iter=1000, random_state=42)
forest = RandomForestClassifier(random_state=42)
lightgbm = LGBMClassifier(random_state=42)
catboost = CatBoostClassifier(iterations=300, cat_features=tuple(categorical_columns), random_seed=42, verbose=False, allow_writing_files=False)


In [ ]:
logistic_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", logistic)
])

forest_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", forest)
])

lightgbm_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", lightgbm)
])

catboost_preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline["imputer"], numeric_columns),
    ("categorical", categorical_pipeline["imputer"], categorical_columns)
], verbose_feature_names_out=False).set_output(transform="pandas")

catboost_pipeline = Pipeline([
    ("preprocessor", catboost_preprocessor),
    ("model", catboost)
])

##### This sets up five-fold cross-validation, keeping each patient’s stays together and defines the same five evaluation metrics for all three models.

In [ ]:
from sklearn.model_selection import cross_validate, GroupKFold

cv = GroupKFold(n_splits=5, shuffle=True, random_state=42)

metrics = ["accuracy", "precision", "recall", "f1", "roc_auc"]

In [ ]:
logistic_scores = cross_validate(
    logistic_pipeline,
    X_train,
    y_train,
    cv=cv,
    groups=groups_train,
    scoring=metrics,
    error_score="raise"
)

logistic_results = pd.DataFrame(logistic_scores)
logistic_results.mean()

In [ ]:
forest_scores = cross_validate(
    forest_pipeline,
    X_train,
    y_train,
    cv=cv,
    groups=groups_train,
    scoring=metrics,
    error_score="raise"
)

forest_results = pd.DataFrame(forest_scores)
forest_results.mean()

In [ ]:
lightgbm_scores = cross_validate(
    lightgbm_pipeline,
    X_train,
    y_train,
    cv=cv,
    groups=groups_train,
    scoring=metrics,
    error_score="raise"
)

lightgbm_results = pd.DataFrame(lightgbm_scores)
lightgbm_results.mean()

In [ ]:
catboost_scores = cross_validate(
    catboost_pipeline,
    X_train,
    y_train,
    cv=cv,
    groups=groups_train,
    scoring=metrics,
    error_score="raise"
)

catboost_results = pd.DataFrame(catboost_scores)
catboost_results.mean()

In [ ]:
comparison = pd.DataFrame({
    "Logistic Regression": logistic_results.filter(like="test_").mean(),
    "Random Forest": forest_results.filter(like="test_").mean(),
    "LightGBM": lightgbm_results.filter(like="test_").mean(),
    "CatBoost": catboost_results.filter(like="test_").mean()
}).T

comparison.round(3)

## 5. Pipeline

### Candidate pipelines

We retain LightGBM and CatBoost for hyperparameter tuning because performances are really similar, and we want to see both performances after tuning.
The final classifier will be selected after tuning.

The LightGBM and CatBoost pipelines reuse the preprocessing already defined in Section 2 and 4, so we do not redefine it here.

In [ ]:
lightgbm_candidate = Pipeline([
    ("preprocessor", preprocessor),
    ("model", lightgbm)
])

catboost_candidate = Pipeline([
    ("preprocessor", catboost_preprocessor),
    ("model", catboost)
])

## 6. Hyperparameter tuning

#### We prepare the tuning for LightGBM

In [ ]:
import optuna

def objective_lightgbm(trial):

    model = LGBMClassifier(
        n_estimators=trial.suggest_int("n_estimators", 100, 500, step=100),
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.2),
        num_leaves=trial.suggest_int("num_leaves", 15, 127),
        min_child_samples=trial.suggest_int("min_child_samples", 20, 100, step=20),
        class_weight=trial.suggest_categorical("class_weight", [None, "balanced"]),
        random_state=42,
        verbosity=-1
    )

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    scores = cross_validate(
        pipeline, X_train, y_train,
        cv=cv,
        groups=groups_train,
        scoring=metrics,
        error_score="raise"
    )

    for name, values in scores.items():
        trial.set_user_attr(name, values.tolist())

    return scores["test_roc_auc"].mean()

In [ ]:
study_lightgbm = optuna.create_study(
    study_name="lightgbm_tuning",
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
    storage="sqlite:///optuna_results.db",
    load_if_exists=True
)

study_lightgbm.optimize(objective_lightgbm, n_trials=50)

In [ ]:
print("Best ROC-AUC:", study_lightgbm.best_value)
print("Best parameters:", study_lightgbm.best_params)

#### Now we also prepare tuning for Catboost, to see which one will be the most performant

In [ ]:
def objective_catboost(trial):

    model = CatBoostClassifier(
        iterations=trial.suggest_int("iterations", 100, 500, step=100),
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.2),
        depth=trial.suggest_int("depth", 4, 8),
        auto_class_weights=trial.suggest_categorical(
            "auto_class_weights", [None, "Balanced"]
        ),
        cat_features=tuple(categorical_columns),
        random_seed=42,
        verbose=False,
        allow_writing_files=False
    )

    pipeline = Pipeline([
        ("preprocessor", catboost_preprocessor),
        ("model", model)
    ])

    scores = cross_validate(
        pipeline, X_train, y_train,
        cv=cv,
        groups=groups_train,
        scoring=metrics,
        error_score="raise"
    )

    for name, values in scores.items():
        trial.set_user_attr(name, values.tolist())

    return scores["test_roc_auc"].mean()

In [ ]:
study_catboost = optuna.create_study(
    study_name="catboost_tuning",
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
    storage="sqlite:///optuna_results.db",
    load_if_exists=True
)

study_catboost.optimize(objective_catboost, n_trials=50)

In [ ]:
print("Best ROC-AUC:", study_catboost.best_value)
print("Best parameters:", study_catboost.best_params)